# 06c · Logistic regression and random forest
Tunes and compares logistic regression and random forest on the provider-level feature tables from notebook 05, using the exact saved split and shared folds from `origin/pei_branch` (reference commit `a8bcd0d`). All variants start from `full_features` (26 columns). Logistic regression compares all features with two fold-local VIF cutoffs; random forest uses all 26.

**Validation protocol**
- **Development (sections 1–10)** reads only the training partition. Test labels are never loaded here; only the test file's header and provider IDs are read for the split audit.
- Three columns that notebook 05 filled with full-dataset means (`TopDiagnosisCodeShare`, `TopProcedureCodeShare`, `BenfordMAD`) are reset to missing using their `…Missing` flags. Within each fold they are then either imputed from that fold's training data.
- Folds come from the **shared fold file**, validated for duplicates, fold coverage and label consistency, so both notebooks use identical folds.
- Every strategy uses the **same fold-local median imputation**, so the comparison isolates the imbalance method. The strategies are: original, fold-local class weighting, and fold-local SMOTE-NC (scaled, with binary flags treated as categorical). A dummy (prior) classifier is the reference floor.
- Primary metric: **average precision (AP)**, reported with fold mean and SD. The threshold and model are chosen on **out-of-fold (OOF)** predictions only, using the agreed threshold rule.
- Each run writes to its own timestamped folder and never overwrites an earlier run.
- Search progress is checkpointed after every configuration.
- **Final test (section 11)** is self-contained. It loads the frozen pipeline and threshold from a full (not quick) run, verifies file fingerprints, scores the held-out set once with bootstrap confidence intervals, and refuses to run a second time.

Do not use notebook 05’s pre-resampled CSVs for CV. Run notebook 05 first if the raw split files are missing. For exact fold alignment, obtain Pei’s existing `artifacts/shared/cv_folds.csv`; this notebook refuses to invent a replacement. CV compares strategies; the test set evaluates only the frozen winner.


**VIF sensitivity comparison:** `lr_all` uses all 26 features, `lr_vif10` selects features with VIF < 10, and `lr_vif5` selects features with VIF < 5. Selection is refitted on each real training fold after median imputation and scaling, before SMOTE. Retained counts may differ across folds; the previous globally selected 22-column list is not used by these variants.


In [ ]:
import hashlib
import json
import platform
import shutil
import time
import warnings
from pathlib import Path

import imblearn
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from imblearn.over_sampling import SMOTE, SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline
from scipy.stats import loguniform, randint, uniform
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (average_precision_score, brier_score_loss, confusion_matrix,
                             f1_score, precision_recall_curve, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import ParameterSampler, StratifiedKFold
from sklearn.preprocessing import StandardScaler

pd.set_option("display.float_format", "{:.4f}".format)
import sys
REPO_ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.fold_vif import FoldVIFSelector, NamedSMOTE


## 1. Configuration
**Threshold rule — agree with Swei before the final run, and write the reason in `THRESHOLD_RATIONALE`:**
- `{"rule": "min_recall", "value": 0.80}` — the highest-precision threshold that still catches ≥ 80% of labelled fraud providers in OOF.
- `{"rule": "review_share", "value": 0.10}` — flag the top 10% highest-scoring providers (a review-capacity limit).
- `{"rule": "f1"}` — maximise F1 (treats precision and recall as equally important).

`QUICK_RUN = True` gives 5 configs per model (300 fits) for checks; `False` gives 20 configs (1,200 fits) for the final search.

**Model selection rule:** a candidate counts as tied with the best when its mean CV AP is within one standard error (`ap_std / √folds`) of the best. Among tied candidates, the simplest imbalance strategy in `STRATEGY_PREFERENCE` wins, then the higher mean AP.
The three LR variants use identical sampled C/penalty settings and tuning budgets. VIF selection lives in `src/fold_vif.py`; keep this module available when loading saved pipelines.


In [ ]:
DATA_DIR      = Path("../data/processed")
TRAIN_PATH    = DATA_DIR / "provider_features_train.csv"
TEST_PATH     = DATA_DIR / "provider_features_test.csv"
FEATURES_PATH = DATA_DIR / "feature_sets.json"
FEATURE_SET   = "full_features"
FOLDS_PATH    = Path("../artifacts/shared/cv_folds.csv")   # shared with Swei
OUT_DIR       = Path("../artifacts/logistic_random_forest")

ID_COL = "Provider"
TARGET = "PotentialFraud"
# Mean-filled upstream with full-data means; reset to NaN via their "<col>Missing" flags
RESTORE_NA_COLS = ["TopDiagnosisCodeShare", "TopProcedureCodeShare", "BenfordMAD"]

SEED = 4012
N_FOLDS = 5
N_JOBS = 4                  # lower this if the laptop struggles; -1 uses every core
QUICK_RUN = False           # True: 5 configs per model; False: 20 (final search)
N_ITER = 5 if QUICK_RUN else 20
STRATEGIES = ["original", "class_weight", "smote"]

THRESHOLD_RULE = {"rule": "min_recall", "value": 0.80}   # TODO: confirm with Swei
# Record WHY this rule was chosen (agreed with Swei, using the section 9 trade-off table).
# The final test (section 11) refuses to run while this is empty.
THRESHOLD_RATIONALE = ""

# Model selection: tie tolerance (in standard errors of the best candidate's fold AP) and strategy preference
SELECTION_TOLERANCE_SE = 1.0
STRATEGY_PREFERENCE = ["original", "class_weight", "smote"]   # simplest first

SPACES = {
    "lr": {
        "C": loguniform(1e-4, 1e2),
        "penalty": ["l1", "l2"],
    },
    "rf": {
        "n_estimators": randint(200, 800),
        "max_depth": [None, 4, 8, 12, 20],
        "min_samples_split": randint(2, 21),
        "min_samples_leaf": randint(1, 16),
        "max_features": ["sqrt", "log2", 0.5, 1.0],
    },
}

LR_VARIANTS = {"lr_all": None, "lr_vif10": 10.0, "lr_vif5": 5.0}
lr_space = SPACES.pop("lr")
SPACES = {**{name: lr_space for name in LR_VARIANTS}, **SPACES}

RUN_ID = time.strftime("%Y%m%d_%H%M%S") + ("_quick" if QUICK_RUN else "_full")
RUN_DIR = OUT_DIR / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)   # never overwrite an earlier run
print(f"Run folder: {RUN_DIR}")

## 2. Load training data and audit the split
Only the training table is loaded. From the test file, only the header and the provider IDs are read, to confirm column availability and zero overlap.

In [ ]:
def load(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if not pd.api.types.is_numeric_dtype(df[TARGET]):
        df[TARGET] = df[TARGET].map({"Yes": 1, "No": 0})
    assert df[TARGET].isin([0, 1]).all(), f"Unexpected target values in {path}"
    assert df[ID_COL].is_unique, f"Duplicate provider IDs in {path}"
    return df


def restore_missing(df: pd.DataFrame, cols: list[str]) -> tuple[pd.DataFrame, dict]:
    df = df.copy()
    counts = {}
    for col in cols:
        flag = f"{col}Missing"
        assert col in df.columns and flag in df.columns, f"Need both {col} and {flag}"
        mask = df[flag] == 1
        df.loc[mask, col] = np.nan
        counts[col] = int(mask.sum())
    return df, counts


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


for required in (TRAIN_PATH, TEST_PATH, FEATURES_PATH, FOLDS_PATH):
    if not required.exists():
        raise FileNotFoundError(f"Missing {required}: run notebook 05 and obtain Pei’s shared folds before tuning.")

train = load(TRAIN_PATH)
train, restored_counts = restore_missing(train, RESTORE_NA_COLS)

test_columns = pd.read_csv(TEST_PATH, nrows=0).columns
test_ids = pd.read_csv(TEST_PATH, usecols=[ID_COL])[ID_COL]
overlap = set(train[ID_COL]) & set(test_ids)
assert not overlap, f"{len(overlap)} providers appear in both train and test"

print(f"Train providers: {len(train)} | fraud providers: {int(train[TARGET].sum())} "
      f"| fraud rate: {train[TARGET].mean():.3%}")
print(f"Test providers (IDs only): {len(test_ids)} | overlap with train: {len(overlap)}")
print("Values reset to NaN (train):", restored_counts)

## 3. Official feature set
Column kinds are inferred from the training data only, without using the label. Binary columns are treated as categorical by SMOTE-NC so it cannot produce flag values like 0.43.

In [ ]:
with open(FEATURES_PATH) as f:
    feature_sets = json.load(f)
features = list(feature_sets[FEATURE_SET])

assert len(features) == len(set(features)), "Duplicate names in feature list"
missing_train = sorted(set(features) - set(train.columns))
missing_test = sorted(set(features) - set(test_columns))
assert not missing_train, f"Missing in train: {missing_train}"
assert not missing_test, f"Missing in test: {missing_test}"
non_numeric = train[features].select_dtypes(exclude="number").columns.tolist()
assert not non_numeric, f"Non-numeric features: {non_numeric}"

X, y = train[features], train[TARGET].to_numpy()


def column_kind(s: pd.Series) -> str:
    v = s.dropna()
    if v.isin([0, 1]).all():
        return "binary"
    if (v >= 0).all() and (v <= 1).all():
        return "proportion"
    if np.allclose(v, np.round(v)):
        return "integer"
    return "continuous"


kinds = {c: column_kind(X[c]) for c in features}
nonneg_cols = [c for c in features if (X[c].dropna() >= 0).all()]
binary_idx = [i for i, c in enumerate(features) if kinds[c] == "binary"]

print(f"{len(features)} features from '{FEATURE_SET}'")
feature_overview = pd.DataFrame({"kind": pd.Series(kinds), "missing_share": X.isna().mean()})
feature_overview.sort_values(["kind", "missing_share"], ascending=[True, False])

logistic_features = list(feature_sets["logistic_features"])
assert set(logistic_features) <= set(features)
assert ID_COL not in features and TARGET not in features
assert train[ID_COL].notna().all() and test_ids.notna().all() and test_ids.is_unique
assert not np.isinf(X.to_numpy(dtype=float)).any(), "Infinite feature values"


## 4. Shared folds
The fold file stores the label as a stale-data check. If Swei already created a fold file, it is loaded and validated rather than replaced.

In [ ]:
fold_df = pd.read_csv(FOLDS_PATH)
print(f"Loaded shared folds from {FOLDS_PATH}")

assert fold_df[ID_COL].is_unique, "Duplicate providers in fold file"
assert fold_df["fold"].notna().all(), "Missing fold values"
assert (fold_df["fold"] == fold_df["fold"].astype(int)).all(), "Non-integer folds"
assert set(fold_df["fold"].astype(int)) == set(range(N_FOLDS)), "Unexpected fold numbers"
assert set(fold_df[ID_COL]) == set(train[ID_COL]), "Fold file does not match training providers"

merged = train[[ID_COL, TARGET]].merge(fold_df, on=ID_COL, how="left",
                                       validate="one_to_one", suffixes=("", "_fold"))
if f"{TARGET}_fold" in merged.columns:
    fold_label = merged[f"{TARGET}_fold"]
    if not pd.api.types.is_numeric_dtype(fold_label):
        fold_label = fold_label.map({"Yes": 1, "No": 0})
    assert (fold_label.to_numpy() == merged[TARGET].to_numpy()).all(), \
        "Labels in fold file differ from training data (stale fold file?)"
else:
    warnings.warn("Fold file has no label column; add PotentialFraud for the stale-data check.")

folds = merged["fold"].to_numpy()
assert not pd.isna(folds).any()
folds = folds.astype(int)

pd.DataFrame({"fold": folds, "y": y}).groupby("fold")["y"].agg(providers="size", fraud="sum", fraud_rate="mean")
for k in range(N_FOLDS):
    assert set(y[folds == k]) == {0, 1}, f"Fold {k} needs both classes"
    assert (y[folds != k] == 1).sum() > 5, "Insufficient fraud rows for SMOTE"


## 5. Fold-local pipelines and VIF sensitivity
Logistic regression variants share regularization tuning and the same folds: `lr_all` (26 features), `lr_vif10`, and `lr_vif5`. Random forest still uses all 26 features.

Each LR pipeline fits median imputation → standardisation → fold-local VIF selection → optional SMOTE-NC → regularized model. VIF is calculated on real training rows, before resampling; validation rows only receive the fitted transformations. Constant columns are removed for VIF variants. The all-feature variant keeps every column. Binary SMOTE indices are resolved by name after selection.

This means SMOTE operates on each variant's retained representation; its synthetic rows may differ across VIF variants. RF retains Pei's full-feature SMOTE representation. Balanced class weights are computed from each training fit's labels.


In [ ]:
def base_model(name: str, params: dict):
    if name in LR_VARIANTS:
        return LogisticRegression(solver="liblinear", max_iter=5000,
                                  random_state=SEED, **params)
    if name == "rf":
        return RandomForestClassifier(random_state=SEED, n_jobs=N_JOBS, **params)
    raise ValueError(name)


def make_sampler():
    if binary_idx:
        return SMOTENC(categorical_features=binary_idx, random_state=SEED)
    return SMOTE(random_state=SEED)


def build(name: str, strategy: str, params: dict, y_fit: np.ndarray):
    if name == "dummy":
        return DummyClassifier(strategy="prior")
    model = base_model(name, params)
    if strategy == "class_weight":
        model.set_params(class_weight="balanced")  # computed from each fit’s labels
    steps = [("impute", SimpleImputer(strategy="median", keep_empty_features=True)
                        .set_output(transform="pandas"))]
    if name in LR_VARIANTS or strategy == "smote":
        steps.append(("scale", StandardScaler().set_output(transform="pandas")))
    if name in LR_VARIANTS:
        steps.append(("select", FoldVIFSelector(threshold=LR_VARIANTS[name])))
    if strategy == "smote":
        steps.append(("smote", NamedSMOTE(
            binary_features=tuple(features[i] for i in binary_idx), random_state=SEED)))
    steps.append(("model", model))
    return ImbPipeline(steps)


def cross_validate(name, strategy, params):
    oof = np.full(len(y), np.nan)
    rows = []
    for k in range(N_FOLDS):
        tr, va = np.where(folds != k)[0], np.where(folds == k)[0]
        est = build(name, strategy, params, y[tr])
        est.fit(X.iloc[tr], y[tr])
        p = est.predict_proba(X.iloc[va])[:, 1]
        oof[va] = p
        rows.append({"fold": k,
                     "ap": average_precision_score(y[va], p),
                     "roc_auc": roc_auc_score(y[va], p)})
        if name in LR_VARIANTS:
            selector = est.named_steps["select"]
            rows[-1].update(n_features=len(selector.selected_features_),
                            selected_features=json.dumps(selector.selected_features_),
                            dropped_features=json.dumps(selector.elimination_log_))
    return oof, pd.DataFrame(rows)


def to_py(d: dict) -> dict:
    return {k: (v.item() if hasattr(v, "item") else v) for k, v in d.items()}

## 6. SMOTE-NC sanity check (one training fold)
This resamples fold 0's training data, maps the synthetic rows back to original units, and checks them. SMOTE interpolates between real rows, so proportions should stay in [0, 1] and non-negative features should stay non-negative. Binary flags should remain exactly 0 or 1. Integer-valued features can become fractional. Tree models can process these values, but the synthetic rows are feature-space approximations, not realistic providers, and they can still shift learned split points.

In [ ]:
if "smote" in STRATEGIES:
    tr0 = np.where(folds != 0)[0]
    X0, y0 = X.iloc[tr0], y[tr0]
    imp = SimpleImputer(strategy="median", keep_empty_features=True).fit(X0)
    scl = StandardScaler().fit(imp.transform(X0))
    X_res, y_res = make_sampler().fit_resample(scl.transform(imp.transform(X0)), y0)
    synth = pd.DataFrame(scl.inverse_transform(X_res[len(X0):]), columns=features)
    print(f"Fold 0: {len(X0)} training rows -> {len(synth)} synthetic minority rows added")

    def share(mask):
        return float(np.mean(mask)) if len(mask) else np.nan

    checks = []
    for c in features:
        v = synth[c].to_numpy()
        k = kinds[c]
        checks.append({
            "feature": c, "kind": k, "synthetic_min": v.min() if len(v) else np.nan,
            "synthetic_max": v.max() if len(v) else np.nan,
            "non_binary_share": share(~np.isclose(v, 0) & ~np.isclose(v, 1)) if k == "binary" else np.nan,
            "fractional_share": share(~np.isclose(v, np.round(v))) if k == "integer" else np.nan,
            "negative_share": share(v < -1e-9) if c in nonneg_cols else np.nan,
            "outside_0_1_share": share((v < -1e-9) | (v > 1 + 1e-9)) if k == "proportion" else np.nan,
        })
    smote_checks = pd.DataFrame(checks)
    smote_checks.to_csv(RUN_DIR / "smote_sanity_checks.csv", index=False)
    problems = smote_checks[(smote_checks[["non_binary_share", "negative_share",
                                           "outside_0_1_share"]].fillna(0) > 0).any(axis=1)]
    print("Impossible values found:" if len(problems) else "No impossible values (flags, negatives, proportions).")
    display(problems if len(problems) else smote_checks[smote_checks["kind"] == "integer"])

## 7. Dummy baseline and random search
Each model draws `N_ITER` configurations, and the **same configurations are run under every imbalance strategy**. For each model × strategy, the configuration with the highest mean CV AP is kept.

In [ ]:
results, best = [], {}

oof_d, fm_d = cross_validate("dummy", "prior", {})
best[("dummy", "prior")] = {"model": "dummy", "strategy": "prior", "config_id": 0, "params": {},
                            "ap_mean": fm_d["ap"].mean(), "ap_std": fm_d["ap"].std(),
                            "roc_auc_mean": fm_d["roc_auc"].mean(), "roc_auc_std": fm_d["roc_auc"].std(),
                            "oof": oof_d, "fold_metrics": fm_d}

total = len(SPACES) * len(STRATEGIES) * N_ITER * N_FOLDS
partial_path = RUN_DIR / "cv_search_results_partial.csv"
print(f"Running {total} model fits (progress saved to {partial_path.name}) ...")
t0 = time.time()
for name, space in SPACES.items():
    configs = [to_py(c) for c in ParameterSampler(space, n_iter=N_ITER, random_state=SEED)]
    for strategy in STRATEGIES:
        for i, params in enumerate(configs):
            oof, fm = cross_validate(name, strategy, params)
            res = {"model": name, "strategy": strategy, "config_id": i,
                   "ap_mean": fm["ap"].mean(), "ap_std": fm["ap"].std(),
                   "roc_auc_mean": fm["roc_auc"].mean(), "roc_auc_std": fm["roc_auc"].std(),
                   "params": json.dumps(params)}
            results.append(res)
            pd.DataFrame(results).to_csv(partial_path, index=False)   # checkpoint
            key = (name, strategy)
            if key not in best or res["ap_mean"] > best[key]["ap_mean"]:
                best[key] = {**res, "params": params, "oof": oof, "fold_metrics": fm}
        b = best[(name, strategy)]
        print(f"{name:5s} {strategy:13s} best AP {b['ap_mean']:.4f} ± {b['ap_std']:.4f}  "
              f"[{time.time() - t0:.0f}s elapsed]")

pd.DataFrame(results).to_csv(RUN_DIR / "cv_search_results.csv", index=False)
partial_path.unlink()   # search completed; the partial checkpoint is no longer needed

## 8. Cross-validation results
Provider-linked OOF probabilities are saved for the dummy and for the best configuration of every model × strategy, ready for Swei's consolidated comparison.

In [ ]:
oof_frames, fold_frames, summary_rows = [], [], []
for (name, strategy), b in best.items():
    oof_frames.append(pd.DataFrame({ID_COL: train[ID_COL], "fold": folds, "y_true": y,
                                    "oof_prob": b["oof"], "model": name, "strategy": strategy}))
    fold_frames.append(b["fold_metrics"].assign(model=name, strategy=strategy))
    summary_rows.append({k: b[k] for k in ["model", "strategy", "config_id", "ap_mean", "ap_std",
                                           "roc_auc_mean", "roc_auc_std"]}
                        | {"pooled_oof_ap": average_precision_score(y, b["oof"])})

pd.concat(oof_frames, ignore_index=True).to_csv(RUN_DIR / "oof_predictions.csv", index=False)
pd.concat(fold_frames, ignore_index=True).to_csv(RUN_DIR / "fold_metrics.csv", index=False)
cv_summary = pd.DataFrame(summary_rows).sort_values("ap_mean", ascending=False).reset_index(drop=True)
cv_summary.to_csv(RUN_DIR / "cv_summary.csv", index=False)
cv_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

labels = [f"{r.model}\n{r.strategy}" for r in cv_summary.itertuples()]
colors = ["#999999" if m == "dummy" else "#4C72B0" for m in cv_summary["model"]]
axes[0].bar(labels, cv_summary["ap_mean"], yerr=cv_summary["ap_std"], capsize=4, color=colors)
axes[0].set_ylabel("Average precision (fold mean ± SD)")
axes[0].set_title("CV average precision by model and imbalance strategy")
axes[0].tick_params(axis="x", labelsize=8)

for (name, strategy), b in best.items():
    prec, rec, _ = precision_recall_curve(y, b["oof"])
    style = "--" if name == "dummy" else "-"
    axes[1].plot(rec, prec, style, label=f"{name}/{strategy} (AP {average_precision_score(y, b['oof']):.3f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Out-of-fold precision–recall curves")
axes[1].legend(fontsize=8)

plt.tight_layout()
fig.savefig(RUN_DIR / "cv_comparison.png", dpi=150)
plt.show()

### Paired fold comparison of imbalance strategies
Every strategy used the same folds, so per-fold differences can be compared directly. For each model, the table shows each strategy's fold AP minus the **original** strategy's fold AP. Folds share training data and each best configuration was chosen on these folds, so this shows how consistent the differences are, not a formal significance test.

In [ ]:
paired_rows = []
for name in SPACES:
    ref = best[(name, "original")]["fold_metrics"].set_index("fold")["ap"]
    for strategy in STRATEGIES:
        if strategy == "original":
            continue
        diff = best[(name, strategy)]["fold_metrics"].set_index("fold")["ap"] - ref
        paired_rows.append({"model": name, "comparison": f"{strategy} − original",
                            **{f"fold_{k}": diff[k] for k in range(N_FOLDS)},
                            "mean_diff": diff.mean(), "folds_worse": int((diff < 0).sum())})
paired = pd.DataFrame(paired_rows)
paired.to_csv(RUN_DIR / "paired_strategy_comparison.csv", index=False)
paired

## 9. Select model and threshold (validation only)
Candidates within `SELECTION_TOLERANCE_SE` standard errors of the best mean CV AP are treated as tied. Among them, the strategy listed first in `STRATEGY_PREFERENCE` is chosen, then the higher mean AP. The dummy is excluded. The threshold comes from the selected candidate's pooled OOF probabilities under `THRESHOLD_RULE`.

In [ ]:
def choose_threshold(y_true, p, rule: dict) -> float:
    kind, val = rule["rule"], rule.get("value")
    if kind == "f1":
        prec, rec, thr = precision_recall_curve(y_true, p)
        f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
        return float(thr[int(np.argmax(f1))])
    if kind == "min_recall":
        if val is None or not 0 < val <= 1:
            raise ValueError(f"min_recall must be in (0, 1], got {val}")
        prec, rec, thr = precision_recall_curve(y_true, p)
        prec, rec = prec[:-1], rec[:-1]
        ok = rec >= val
        if not ok.any():
            raise ValueError(f"No threshold reaches recall {val}")
        return float(thr[int(np.argmax(np.where(ok, prec, -1)))])
    if kind == "review_share":
        if val is None or not 0 < val < 1:
            raise ValueError(f"review_share must be in (0, 1), got {val}")
        return float(np.quantile(p, 1 - val))
    raise ValueError(f"Unknown threshold rule: {kind}")


def threshold_metrics(y_true, p, t) -> dict:
    yhat = (p >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, yhat, labels=[0, 1]).ravel()
    return {"threshold": float(t),
            "ap": float(average_precision_score(y_true, p)),
            "roc_auc": float(roc_auc_score(y_true, p)),
            "precision": float(precision_score(y_true, yhat, zero_division=0)),
            "recall": float(recall_score(y_true, yhat, zero_division=0)),
            "f1": float(f1_score(y_true, yhat, zero_division=0)),
            "flagged_share": float(yhat.mean()),
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}


candidates = cv_summary[cv_summary["model"] != "dummy"].reset_index(drop=True)
top = candidates.loc[0]
selection_se = float(top["ap_std"] / np.sqrt(N_FOLDS))
selection_cutoff = float(top["ap_mean"] - SELECTION_TOLERANCE_SE * selection_se)
candidates["tied_with_best"] = candidates["ap_mean"] >= selection_cutoff
candidates["strategy_rank"] = candidates["strategy"].map({s: i for i, s in enumerate(STRATEGY_PREFERENCE)})
tied = candidates[candidates["tied_with_best"]].sort_values(["strategy_rank", "ap_mean"], ascending=[True, False])
sel_key = (tied.iloc[0]["model"], tied.iloc[0]["strategy"])
sel = best[sel_key]

print(f"Best mean CV AP {top['ap_mean']:.4f}; tie cutoff {selection_cutoff:.4f} "
      f"({SELECTION_TOLERANCE_SE} SE = {SELECTION_TOLERANCE_SE * selection_se:.4f})")
display(candidates[["model", "strategy", "ap_mean", "ap_std", "tied_with_best"]])
threshold = choose_threshold(y, sel["oof"], THRESHOLD_RULE)
oof_metrics = threshold_metrics(y, sel["oof"], threshold)

print(f"Selected: {sel_key[0]} / {sel_key[1]}  (config {sel['config_id']})")
print("Params:", sel["params"])
print("Threshold rule:", THRESHOLD_RULE)

comparison_oof = pd.DataFrame({
    "selected (OOF)": oof_metrics,
    "dummy (OOF, t=0.5)": threshold_metrics(y, best[("dummy", "prior")]["oof"], 0.5),
})
comparison_oof

In [ ]:
# Fold-level metrics for the selected model at the chosen threshold
sel_fold_metrics = pd.DataFrame([
    {"fold": k, **threshold_metrics(y[folds == k], sel["oof"][folds == k], threshold)}
    for k in range(N_FOLDS)
])
sel_fold_metrics.to_csv(RUN_DIR / "selected_fold_metrics.csv", index=False)
cols = ["ap", "roc_auc", "precision", "recall", "f1", "flagged_share"]
display(sel_fold_metrics[["fold"] + cols])
sel_fold_metrics[cols].agg(["mean", "std"])

In [ ]:
# Threshold trade-off table (OOF) to discuss with Swei
grid = np.round(np.arange(0.05, 0.96, 0.05), 2)
tradeoff = pd.DataFrame([threshold_metrics(y, sel["oof"], t) for t in grid])
tradeoff.to_csv(RUN_DIR / "threshold_tradeoff_oof.csv", index=False)
tradeoff[["threshold", "precision", "recall", "f1", "flagged_share", "tp", "fp", "fn"]]

### Calibration
Class weighting and SMOTE shift predicted probabilities upward, so raw scores may not be true probabilities. If the reliability curve sits far from the diagonal, the dashboard should call them **risk scores** and should not describe them as "probability of fraud".

In [ ]:
brier = brier_score_loss(y, sel["oof"])
brier_ref = brier_score_loss(y, np.full(len(y), y.mean()))
prob_true, prob_pred = calibration_curve(y, sel["oof"], n_bins=10, strategy="quantile")
print(f"Brier score: {brier:.4f} (prior-only reference {brier_ref:.4f})")
print(f"Mean OOF score {sel['oof'].mean():.3f} vs actual fraud rate {y.mean():.3f}")

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], "--", c="grey", label="perfect calibration")
ax.plot(prob_pred, prob_true, "o-", label=f"{sel_key[0]}/{sel_key[1]}")
ax.set_xlabel("Mean predicted score (bin)")
ax.set_ylabel("Observed fraud rate (bin)")
ax.set_title("Reliability diagram (OOF)")
ax.legend()
fig.savefig(RUN_DIR / "calibration_oof.png", dpi=150)
plt.show()

## 10. Refit on the full training partition and freeze the run
This saves the pipeline, metadata (feature order, parameters, threshold, seeds, versions and NaN-restoration columns for Streamlit), file fingerprints, and copies of the fold file and feature list.

In [ ]:
final_pipeline = build(sel_key[0], sel_key[1], sel["params"], y)
final_pipeline.fit(X, y)
joblib.dump(final_pipeline, RUN_DIR / "selected_pipeline.joblib")
if not THRESHOLD_RATIONALE.strip():
    warnings.warn("THRESHOLD_RATIONALE is empty; this run cannot be used for the final test.")

shutil.copy(FOLDS_PATH, RUN_DIR / "cv_folds.csv")
shutil.copy(FEATURES_PATH, RUN_DIR / "feature_sets.json")

metadata = {
    "run_id": RUN_ID, "quick_run": QUICK_RUN,
    "model": sel_key[0], "strategy": sel_key[1], "config_id": int(sel["config_id"]),
    "params": sel["params"],
    "threshold": threshold, "threshold_rule": THRESHOLD_RULE,
    "threshold_rationale": THRESHOLD_RATIONALE,
    "selection_rule": {"tolerance_se": SELECTION_TOLERANCE_SE, "strategy_preference": STRATEGY_PREFERENCE,
                       "best_ap_mean": float(top["ap_mean"]), "cutoff": selection_cutoff,
                       "tied_candidates": [f"{m}/{st}" for m, st in zip(tied["model"], tied["strategy"])]},
    "oof_metrics": oof_metrics,
    "cv_ap_mean": float(sel["ap_mean"]), "cv_ap_std": float(sel["ap_std"]),
    "cv_roc_auc_mean": float(sel["roc_auc_mean"]), "cv_roc_auc_std": float(sel["roc_auc_std"]),
    "oof_brier": float(brier), "oof_brier_prior_reference": float(brier_ref),
    "feature_set": FEATURE_SET, "feature_order": features,
    "model_features": (final_pipeline.named_steps["select"].selected_features_
                       if sel_key[0] in LR_VARIANTS else features),
    "vif_threshold": LR_VARIANTS.get(sel_key[0]),
    "alignment_reference": "origin/pei_branch@a8bcd0d",
    "binary_features": [features[i] for i in binary_idx],
    "restore_na_cols": RESTORE_NA_COLS,
    "id_col": ID_COL, "target": TARGET,
    "train_path": str(TRAIN_PATH), "test_path": str(TEST_PATH),
    "n_train": int(len(train)), "n_train_fraud": int(y.sum()), "train_fraud_rate": float(y.mean()),
    "seed": SEED, "n_folds": N_FOLDS, "n_iter": N_ITER, "n_jobs": N_JOBS, "strategies": STRATEGIES,
    "fingerprints_sha256": {"train": sha256(TRAIN_PATH), "test": sha256(TEST_PATH),
                            "feature_sets": sha256(FEATURES_PATH), "folds": sha256(FOLDS_PATH),
                            "pipeline": sha256(RUN_DIR / "selected_pipeline.joblib")},
    "versions": {"python": platform.python_version(), "pandas": pd.__version__,
                 "numpy": np.__version__, "sklearn": sklearn.__version__,
                 "imblearn": imblearn.__version__},
}
with open(RUN_DIR / "selected_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print(f"Frozen run saved to {RUN_DIR.resolve()}")
print(f'For the final test, set TEST_RUN_ID = "{RUN_ID}" in section 11.')

## 11. Shared final evaluation and saved results

The final team evaluation is complete: **06b LightGBM / original**, run `20261011_232303_full`. The next cell contains the guarded evaluation entry point and stays disabled. The following read-only cell displays the saved final result without opening the raw test table or rerunning predictions.

Both notebooks use `src/final_evaluation.py`. Evaluation refuses unless the complete run path, threshold and pipeline fingerprint match `artifacts/shared/final_model_selection.json`, and refuses if a shared evaluation has already started/completed or held-out outputs exist. Do not remove the shared records to bypass this guard.

The threshold rationale was added to the frozen run metadata before the original test function was called; the validation-selected threshold and fitted model were unchanged. The original executed code is archived under `scripts/provenance/`. See `reports/final_model_evaluation.md` for the decision and limitations.


In [ ]:
from pathlib import Path
import sys

EVAL_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "src/final_evaluation.py").is_file()), None)
if EVAL_ROOT is None:
    raise FileNotFoundError("Open this notebook inside the repository containing src/final_evaluation.py.")
if str(EVAL_ROOT) not in sys.path:
    sys.path.insert(0, str(EVAL_ROOT))

from src.final_evaluation import run_selected_test

TEST_RUN_ID = ""
I_CONFIRM_FINAL = False  # leave disabled: the shared final evaluation is complete
RUNS_DIR = EVAL_ROOT / "artifacts/logistic_random_forest"

if I_CONFIRM_FINAL and TEST_RUN_ID:
    final_results = run_selected_test(EVAL_ROOT, RUNS_DIR / TEST_RUN_ID)
    display(final_results)
else:
    print("Evaluation is disabled here. Read the saved final team results in the next cell.")


### Saved final team result (read only)
This is the selected **06b LightGBM** evaluation, including when shown in 06c. It is not a separate test of the 06c winner. Re-running this display cell only reads saved evidence.


In [ ]:
from pathlib import Path
import sys

EVAL_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "src/final_evaluation.py").is_file()), None)
if EVAL_ROOT is None:
    raise FileNotFoundError("Open this notebook inside the repository containing src/final_evaluation.py.")
if str(EVAL_ROOT) not in sys.path:
    sys.path.insert(0, str(EVAL_ROOT))

import pandas as pd
from IPython.display import display
from src.final_evaluation import load_saved_results

# Read-only: no model loading, test prediction, training or bootstrap execution.
saved_final = load_saved_results(EVAL_ROOT)
selection = saved_final["selection"]
test_result = saved_final["test"]
print(f"FINAL TEAM RESULT — {selection['model']} / {selection['strategy']} ({selection['source_notebook']})")
print(f"Selected run: {selection['selected_run']}")
print("Loaded saved results only. The held-out evaluation is already complete.")
rows = []
for metric in ["ap", "roc_auc", "precision", "recall", "f1"]:
    interval = test_result["bootstrap_95ci"][metric]
    rows.append({"metric": metric, "validation_OOF": selection["oof_metrics"][metric],
                 "held_out_test": test_result["point"][metric],
                 "test_CI_low": interval["ci_low"], "test_CI_high": interval["ci_high"]})
display(pd.DataFrame(rows).set_index("metric"))
p = test_result["point"]
print(f"TP={p['tp']}, FP={p['fp']}, FN={p['fn']}, TN={p['tn']}; "
      f"flagged={p['tp'] + p['fp']}/{test_result['n_test']}; threshold={selection['threshold']:.12f}")


FINAL TEAM RESULT — lgbm / original (06b)
Selected run: artifacts/boosting/20261011_232303_full
Loaded saved results only. The held-out evaluation is already complete.
TP=86, FP=91, FN=15, TN=890; flagged=177/1082; threshold=0.086104647726


,validation_OOF,held_out_test,test_CI_low,test_CI_high
metric,,,,
ap,0.671979,0.753604,0.679533,0.826461
roc_auc,0.935268,0.960043,0.944307,0.972903
precision,0.462197,0.485876,0.434777,0.539406
recall,0.800000,0.851485,0.772277,0.920792
f1,0.585895,0.618705,0.564456,0.671481


## 12. Interpretation and limitations
Compare `lr_all`, `lr_vif10`, and `lr_vif5` within each imbalance strategy using mean fold AP and SD. Fold metrics also record retained feature names, counts, and VIF elimination logs. Feature counts need not be 22: selection is learned from each fold rather than the full dataset.

The same regularization configurations and provider folds are used for all LR variants. Strategy selection and the guarded final test remain unchanged; test labels must not influence the VIF cutoff. The full search now has 1,200 model fits plus the dummy baseline and final refit.

CV is optimistic because these folds are used for tuning and model selection. Upstream full-data feature development remains a limitation. SMOTE runs after VIF selection, so its geometry can differ across variants. Coefficient stability is not measured automatically by this comparison; AP alone does not establish interpretability. Keep `src/fold_vif.py` available when loading saved pipelines.
